# License and Attribution

**Copyright © 2026 Randy Balzer. All rights reserved.**

This notebook is shared publicly via GitHub for educational, research, and professional portfolio purposes.

**Attribution Requirement**  
If any portion of this work is used, adapted, extended, or incorporated into other projects, research, presentations, or commercial applications, clear and prominent attribution is required in the following form:

> Based on work by Randy Balzer (randy@balzer.io). Original notebook: *agentic_code_vulnerability_analysis_poc.ipynb*.

For substantial derivative works or any commercial application, please contact randy@balzer.io to discuss appropriate licensing terms.

This material is provided “as is” without warranty of any kind, express or implied. The author disclaims all liability arising from the use or misuse of these notebooks.

# Agentic Code Vulnerability Analysis PoC

**Multi-Agent Framework for Automated Code Vulnerability Review**

This notebook demonstrates a lightweight multi-agent pipeline using LangChain that:
1. Reviews source code for common security vulnerabilities
2. Summarizes findings with severity and remediation guidance
3. Orchestrates the review process through a supervisor pattern

Designed as a focused proof-of-concept aligned with research on LLM-based vulnerability detection and hybrid human-in-the-loop analysis.

## Overview

**What this notebook demonstrates**
- A multi-agent approach to automated code vulnerability analysis
- Specialized agents for code review and structured reporting
- Practical recognition of LLM strengths and limitations for vulnerability detection
- Alignment with hybrid systems (automated analysis + human judgment)

**Key points**
- Complements the CVE impact analysis notebook (discovery + code-level analysis)
- Directly relevant to AI-augmented static analysis and offensive research
- Acknowledges limitations of standalone LLM detection (false positives, missed context)
- Designed for extension into real SCA / SAST integration

**Limitations / scaling challenges**
- LLMs can hallucinate findings or miss subtle context-dependent issues
- Best used as an assistive layer, not a replacement for traditional SAST + human review
- Production use requires grounding, validation, and human-in-the-loop gates

## 1. Setup, Imports, and Configuration

In [3]:
!pip install python-dotenv langchain-xai langchain langchain-core -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # loads .env from project root

import json
import textwrap
from typing import Dict, Any, List

from IPython.display import display, Markdown

from langchain_xai import ChatXAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load XAI API Key securely
if not os.environ.get("XAI_API_KEY"):
    os.environ["XAI_API_KEY"] = getpass("Enter your XAI API Key: ")

print("XAI API Key loaded.")

llm = ChatXAI(
    model="grok-4",
    temperature=0.1,
    max_tokens=800
)

XAI API Key loaded.


## 2. Sample Vulnerable Code Snippets

A small set of intentionally vulnerable examples for demonstration. In a real system these would come from a repository or pull request.

In [5]:
SAMPLE_FILES = {
    "login.py": '''
import sqlite3
from flask import request

def authenticate(username, password):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    # Vulnerable: string concatenation → SQL injection
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    cursor.execute(query)
    result = cursor.fetchone()
    conn.close()
    return result is not None

def get_user_profile(user_id):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    query = "SELECT * FROM users WHERE id = " + user_id   # also vulnerable
    cursor.execute(query)
    return cursor.fetchone()
''',

    "file_handler.py": '''
import os
from flask import request, send_file

def download_report():
    filename = request.args.get("file")
    # Vulnerable: path traversal
    path = "/var/reports/" + filename
    return send_file(path)

def save_upload():
    f = request.files["upload"]
    # Vulnerable: unrestricted file upload + path traversal
    f.save("/tmp/uploads/" + f.filename)
    return "ok"
''',

    "config.py": '''
import os

# Hardcoded secrets (bad practice)
DB_PASSWORD = "SuperSecretPassword123!"
API_KEY = "sk-live-abc123xyz789"
AWS_SECRET = "wJalrXUtnFEMI/K7MDENG/bPxRfiCYEXAMPLEKEY"

def get_db_connection():
    import psycopg2
    return psycopg2.connect(
        host="prod-db.internal",
        user="admin",
        password=DB_PASSWORD
    )
''',

    "safe_example.py": '''
import sqlite3
from flask import request

def authenticate(username, password):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    # Parameterized query — safer
    cursor.execute(
        "SELECT * FROM users WHERE username = ? AND password = ?",
        (username, password)
    )
    result = cursor.fetchone()
    conn.close()
    return result is not None
'''
}

print(f"Loaded {len(SAMPLE_FILES)} sample files")
for name in SAMPLE_FILES:
    print(f"  - {name}")

Loaded 4 sample files
  - login.py
  - file_handler.py
  - config.py
  - safe_example.py


## 3. Agent 1 — Code Review Agent

Performs a structured security review of a source file and returns findings in a consistent JSON-like format.

In [6]:
code_review_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert application security engineer performing a focused code review.
Analyze the provided source code for common security vulnerabilities.

Look specifically for:
- SQL injection / command injection
- Path traversal / arbitrary file access
- Hardcoded secrets or credentials
- Insecure deserialization
- XSS / unsafe output encoding (if web-related)
- Authentication or authorization flaws
- Use of dangerous functions or patterns

For each finding return:
- vulnerability_type
- severity (Critical / High / Medium / Low)
- location (function or approximate lines)
- description
- evidence (short code snippet)
- recommendation

Return ONLY valid JSON in this format:
[
  {{
    "vulnerability_type": "...",
    "severity": "Critical|High|Medium|Low",
    "location": "...",
    "description": "...",
    "evidence": "...",
    "recommendation": "..."
  }}
]
If no issues are found, return an empty list [].
Be precise. Do not invent vulnerabilities that are not present."""),
    ("human", """Filename: {filename}

Source code:
```
{code}
```

Return the vulnerability findings as JSON.""")
])

code_reviewer = code_review_prompt | llm | StrOutputParser()

def review_code(filename: str, code: str) -> list:
    result = code_reviewer.invoke({
        "filename": filename,
        "code": code
    })
    try:
        cleaned = result.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        return json.loads(cleaned)
    except Exception as e:
        print(f"Parse error in code review agent: {e}")
        print("Raw output:", result)
        return []

## 4. Agent 2 — Vulnerability Reporter Agent

Takes raw findings from the Code Review Agent and produces a clear, prioritized human-readable report.

In [ ]:
reporter_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior security analyst writing a concise vulnerability report for engineering and leadership audiences.
Given a list of findings from a code review, produce a clear, prioritized report.

Structure the report as:
1. Executive Summary (2–3 sentences)
2. Findings by Severity
3. Recommended Next Actions
4. Notes / Limitations (brief)

Use clear language. Highlight the highest-risk issues first.
If there are no findings, state that clearly."""),
    ("human", """Filename: {filename}

Raw findings:
{findings_json}

Produce the vulnerability report.""")
])

reporter = reporter_prompt | llm | StrOutputParser()

def generate_report(filename: str, findings: list) -> str:

    return reporter.invoke({
        "filename": filename,
        "findings_json": json.dumps(findings, indent=2)
    })
    

## 5. Supervisor / Orchestrator

Runs the full multi-agent review pipeline on one or more files and presents the results cleanly.

**Important:** Run all cells above this one first (Setup → Agents 1–2).

In [ ]:
def _wrap(text: str, width: int = 88) -> str:
    return textwrap.fill(text, width=width)

def run_code_vulnerability_pipeline(filenames: list = None) -> None:
    
    if filenames is None:
        filenames = list(SAMPLE_FILES.keys())

    print("=" * 70)
    print("AGENTIC CODE VULNERABILITY ANALYSIS PIPELINE")
    print("=" * 70)

    all_findings = {}

    for filename in filenames:
        if filename not in SAMPLE_FILES:
            print(f"\n[!] Skipping unknown file: {filename}")
            continue

        code = SAMPLE_FILES[filename]
        print(f"\n{'─' * 70}")
        print(f"Reviewing: {filename}")
        print(f"{'─' * 70}")

        # Agent 1: Code Review
        print("\n[Agent 1] Code Review Agent — analyzing source...")
        findings = review_code(filename, code)
        all_findings[filename] = findings

        if findings:
            print(f"  → Found {len(findings)} potential issue(s):")
            for f in findings:
                print(f"     • [{f.get('severity')}] {f.get('vulnerability_type')} — {f.get('location')}")
        else:
            print("  → No significant issues identified.")

        # Agent 2: Reporter
        print("\n[Agent 2] Vulnerability Reporter — generating report...")
        report = generate_report(filename, findings)

        display(Markdown(f"### Report: `{filename}`\n\n{report}"))

    # Final summary across files
    total = sum(len(v) for v in all_findings.values())
    print("\n" + "=" * 70)
    print(f"Pipeline complete. Reviewed {len(filenames)} file(s), {total} total finding(s).")
    print("=" * 70)

# Run on all sample files
run_code_vulnerability_pipeline()

AGENTIC CODE VULNERABILITY ANALYSIS PIPELINE

──────────────────────────────────────────────────────────────────────
Reviewing: login.py
──────────────────────────────────────────────────────────────────────

[Agent 1] Code Review Agent — analyzing source...
  → Found 2 potential issue(s):
     • [Critical] SQL Injection — authenticate
     • [Critical] SQL Injection — get_user_profile

[Agent 2] Vulnerability Reporter — generating report...


### Report: `login.py`

**1. Executive Summary**

The `login.py` module contains two critical SQL injection vulnerabilities that allow attackers to manipulate database queries through unsanitized user input. Both issues stem from direct string concatenation of user-controlled values into SQL statements, exposing the application to data exfiltration, authentication bypass, and potential data modification. Immediate remediation is required before the code is deployed or exposed to untrusted input.

**2. Findings by Severity**

**Critical**

- **SQL Injection in `authenticate`**  
  Unsanitized username and password values are concatenated into an SQL query using an f-string, enabling attackers to alter the query logic (e.g., bypass authentication).  
  *Evidence*: `query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"`

- **SQL Injection in `get_user_profile`**  
  User-controlled `user_id` is directly concatenated into an SQL query, allowing arbitrary query manipulation and potential data exposure.  
  *Evidence*: `query = "SELECT * FROM users WHERE id = " + user_id`

**3. Recommended Next Actions**

- Replace both vulnerable queries with parameterized statements (prepared statements) as specified in the findings.
- Conduct a broader search across the codebase for similar string-concatenation patterns in database calls.
- Add input validation and least-privilege database accounts as defense-in-depth measures.
- Perform a security-focused test (including attempted injection payloads) before release.

**4. Notes / Limitations**

Report is based solely on the provided static code review findings; no runtime testing or additional files were reviewed.


──────────────────────────────────────────────────────────────────────
Reviewing: file_handler.py
──────────────────────────────────────────────────────────────────────

[Agent 1] Code Review Agent — analyzing source...
  → Found 2 potential issue(s):
     • [High] Path Traversal / Arbitrary File Access — download_report (lines 5-7)
     • [High] Path Traversal + Unrestricted File Upload — save_upload (lines 10-12)

[Agent 2] Vulnerability Reporter — generating report...


### Report: `file_handler.py`

**1. Executive Summary**

The review of `file_handler.py` identified two high-severity path traversal vulnerabilities that enable arbitrary file read and write operations. These issues allow unauthenticated or low-privileged users to access files outside intended directories and to upload malicious content, creating risks of data exfiltration, system compromise, and unauthorized code execution. Immediate remediation of input handling and file operations is required.

**2. Findings by Severity**

**High**

- **Path Traversal / Arbitrary File Access** (`download_report`, lines 5-7)  
  Unsanitized `request.args.get("file")` is concatenated directly into a path and passed to `send_file`, permitting access to any file on the system.  
  *Evidence*: `filename = request.args.get("file"); path = "/var/reports/" + filename; return send_file(path)`

- **Path Traversal + Unrestricted File Upload** (`save_upload`, lines 10-12)  
  User-supplied `f.filename` is used without sanitization in `f.save`, enabling both directory traversal writes and storage of arbitrary/malicious files.  
  *Evidence*: `f = request.files["upload"]; f.save("/tmp/uploads/" + f.filename)`

**3. Recommended Next Actions**

- Apply `werkzeug.utils.secure_filename` (or equivalent) and resolve/validate the final path against an allow-list base directory for both download and upload functions.
- Replace user-controlled filenames with server-generated names and enforce strict content-type and size validation on uploads.
- Add unit tests covering path traversal payloads and schedule a follow-up review after fixes are implemented.

**4. Notes / Limitations**

Findings are based solely on static code review of the provided snippets; no runtime testing or full application context was available.


──────────────────────────────────────────────────────────────────────
Reviewing: config.py
──────────────────────────────────────────────────────────────────────

[Agent 1] Code Review Agent — analyzing source...
  → Found 1 potential issue(s):
     • [High] Hardcoded secrets or credentials — config.py (top-level variables and get_db_connection)

[Agent 2] Vulnerability Reporter — generating report...


### Report: `config.py`

**1. Executive Summary**

A single high-severity finding was identified in `config.py`: multiple sensitive credentials (database password, API key, and AWS secret access key) are hardcoded in source code. This exposes the application to credential leakage through code repositories, logs, or supply-chain attacks and violates basic secrets-management practices. Immediate remediation is required to prevent unauthorized access to production systems and data.

**2. Findings by Severity**

**High**
- **Hardcoded secrets or credentials** (`config.py`, top-level variables and `get_db_connection`)
  - Database password, live API key, and AWS secret key are stored directly in the source file.
  - Evidence includes `DB_PASSWORD = "SuperSecretPassword123!"`, `API_KEY = "sk-live-abc123xyz789"`, and `AWS_SECRET = "wJalrXUtnFEMI/K7MDENG/bPxRfiCYEXAMPLEKEY"`.
  - Risk: Full compromise of database, external APIs, and AWS resources if the code is accessed by unauthorized parties.

**3. Recommended Next Actions**

- Remove all hardcoded credentials from `config.py` and replace them with references to environment variables or a secrets-management service (AWS Secrets Manager, HashiCorp Vault, or Azure Key Vault).
- Rotate the exposed credentials immediately.
- Add a pre-commit hook or CI check (e.g., `detect-secrets`, `trufflehog`) to prevent future commits of secrets.
- Conduct a broader secrets scan across the entire codebase and Git history.

**4. Notes / Limitations**

Report is based solely on the single provided finding in `config.py`; no other files were reviewed.


──────────────────────────────────────────────────────────────────────
Reviewing: safe_example.py
──────────────────────────────────────────────────────────────────────

[Agent 1] Code Review Agent — analyzing source...
  → No significant issues identified.

[Agent 2] Vulnerability Reporter — generating report...


### Report: `safe_example.py`

**1. Executive Summary**

The code review of `safe_example.py` identified no security vulnerabilities, weaknesses, or risky patterns. The file appears free of common issues such as injection flaws, insecure data handling, or unsafe dependencies. No immediate remediation is required.

**2. Findings by Severity**

- **Critical / High / Medium / Low**: None  
- No findings were reported.

**3. Recommended Next Actions**

- Maintain current secure coding practices.  
- Continue periodic reviews as the codebase evolves.

**4. Notes / Limitations**

Review was limited to the provided static findings list; dynamic analysis or runtime testing was not performed.


Pipeline complete. Reviewed 4 file(s), 5 total finding(s).


## 6. Review a Single File

You can also target one file at a time.

In [ ]:
# Example: review only the login module
# run_code_vulnerability_pipeline(["login.py"])

# Or the relatively safe example
# run_code_vulnerability_pipeline(["safe_example.py"])

## Notes for Extension / Research

- Replace sample files with real repository content or pull-request diffs.
- Add a third agent for cross-file / data-flow analysis (taint-style reasoning).
- Ground findings with traditional SAST tools (Semgrep, CodeQL, Bandit) to reduce hallucinations.
- Add human-in-the-loop confirmation before findings are accepted.
- This pattern supports research into hybrid multi-agent systems for automated vulnerability detection, consistent with findings that pure LLM detectors remain limited in reliability.